# HachimiMT — Dịch Trung → Việt (Kaggle, GPU T4 miễn phí)

Chạy app dịch truyện **HachimiMT / MoxhiMT** (CTranslate2) trên Kaggle — GPU T4 miễn phí (~30 giờ/tuần), không cần cài gì trên máy.

## ⚡ Trước khi chạy — bật GPU + Internet
1. **`Settings → Accelerator → GPU T4 x2`** (app tự dùng các GPU CUDA khả dụng; T4 x2 thường nhanh hơn 1 T4 khi văn bản đủ dài).
2. **`Settings → Internet → on`** (BẮT BUỘC — để tải code + model và tạo link công khai).
3. Rồi **`Run All`**.

> ⚠️ **Chọn `GPU T4 x2`** — **KHÔNG chọn `GPU P100`**: P100 (kiến trúc cũ) không hỗ trợ kiểu tính `int8_float16` của model nên app sẽ rớt về CPU (chậm ~50×). Cũng **không chọn TPU** (CTranslate2 không dùng được TPU).

### 🚀 Tốc độ đo thật gần nhất (T4 x2, HachimiMT-60 CT2)
| Chế độ | Tốc độ | File 2,84M chữ |
|---|---|---|
| **Beam 1** (ưu tiên tốc độ) | ~71.000 chữ Hán/giây | ~40 giây |
| **Beam 2** (mặc định, chất lượng) | ~52.000 chữ Hán/giây | ~55 giây |

> Beam thấp dịch nhanh hơn nhưng chất lượng nhỉnh kém hơn chút (chọn Beam trong giao diện). Colab/T4 x1 đã tối ưu đạt ~28.000 chữ/giây ở beam 2 hoặc ~40.000 ở beam 1; Kaggle T4 x2 vẫn nhanh nhất khi dịch file dài.

Cell cuối in ra **link công khai** (`*.gradio.live`) — bấm vào đó để mở giao diện dịch.

Mã nguồn: HF Space [ngocdang83/HachimiMT-demo](https://huggingface.co/spaces/ngocdang83/HachimiMT-demo).

### 1. Tải mã nguồn app (từ HF Space, ~75 KB)

In [ ]:
import urllib.request, zipfile, os, shutil

ZIP_URL = "https://huggingface.co/spaces/ngocdang83/HachimiMT-demo/resolve/main/hachimimt-local.zip"
urllib.request.urlretrieve(ZIP_URL, "hachimimt-local.zip")
shutil.rmtree("hachimimt", ignore_errors=True)   # xóa bản cũ nếu chạy lại
with zipfile.ZipFile("hachimimt-local.zip") as z:
    z.extractall(".")
print("Đã tải + giải nén:", sorted(os.listdir("hachimimt")))

### 2. Cài thư viện (CTranslate2 + Gradio + tokenizer) — ~1 phút

> Nếu thấy vài dòng đỏ `ERROR: pip's dependency resolver ...` về gói Kaggle cài sẵn — **vô hại**, cứ chạy tiếp.

In [ ]:
!pip install -q -r hachimimt/requirements.txt

### 3. Chạy app — đợi link `*.gradio.live` hiện ra rồi bấm vào

Lần dịch đầu sẽ tự tải model (~57 MB) từ Hugging Face.

In [ ]:
import os, sys

# Bật link share công khai (gradio.live) — app đọc cờ này (cần Internet=on).
os.environ["HACHIMIMT_SHARE"] = "1"
# Kaggle T4 x2 đồng cấu hình: cho app tự dùng cả 2 GPU nếu có.
os.environ["HACHIMIMT_AUTO_ALL_GPUS"] = "1"
sys.path.insert(0, os.path.abspath("hachimimt/src"))

# Báo rõ đang chạy GPU hay CPU.
from hardware import detect_hardware_profile
_hw = detect_hardware_profile()
if _hw.has_cuda:
    print(f"✅ Đang dùng GPU: {_hw.gpu_name} — nhanh!  ({_hw.summary})")
else:
    print("⚠️ Đang chạy CPU (chậm ~50× GPU). Kiểm tra:\n"
          "   • Settings → Accelerator → GPU T4 x2 (KHÔNG chọn P100 — không tương thích, "
          "sẽ rớt CPU)\n"
          "   • Settings → Internet → on\n"
          "   rồi Run All lại.")

import app
app.main()